# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset and print the metadata summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

### Additional Metadata
The dataset was collected using a structured survey involving 475 pastoralist households across four wards in Samburu, Isiolo, and Marsabit counties, Northern Kenya. It contains ordered logistic regression outputs, socio-demographic characteristics, and intervention outcome data. 

- **Keywords:** adoption predictors, climate adaptation, extension services, gender inclusion, indigenous knowledge
- **Spatial Coverage:** Samburu, Isiolo, Marsabit (Northern Kenya)
- **Temporal Coverage:** 2021-11-16 to 2024-11-16
- **Data Biases & Limitations:** Male respondents are overrepresented. There are missing values in some fields.

Let's continue to explore the available record sets in this dataset.

## 2. Data Overview
Review available record sets, fields, and IDs.

We'll list all the available record sets and preview their fields using their `@id` fields for reference. This helps identify which data tables and columns are present for further analysis.

In [ ]:
# List and describe all RecordSets by @id
record_sets = dataset.record_sets
print(f"{len(record_sets)} record set(s) found.\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}")
    print("  Fields (with @id):")
    for field in rs.fields:
        print(f"   - {field.name} (@id: {field.id})")
    print('-'*50)

## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrames for analysis. 

We'll use the record set and field `@id` values from above to reference the data unambiguously. Each record set is loaded separately.

In [ ]:
# Gather all record set @id values for bulk extraction
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # The generator yields each record as a dict keyed by field @id
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if records exist
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()  # Empty placeholder

# For demonstration, pick the first non-empty record set to analyze further
selected_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rid
        break

if selected_record_set_id:
    print(f"Sample columns for record set {selected_record_set_id}:\n",
          dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping. Make sure to reference fields by their `@id`.

As an example, we'll:
- Choose a numeric field
- Filter rows above a threshold
- Normalize values
- Group by a categorical field (if present)

***Note:*** The names and IDs below are illustrative; please adapt them for your specific analysis, matching them to fields discovered above.

In [ ]:
# EDA on the selected record set
df = dataframes[selected_record_set_id]

if df.empty:
    print("No data available for EDA.")
else:
    # Attempt to choose a numeric field by looking for float/int dtype
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric fields detected for EDA.")
    else:
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} (@id) > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df = filtered_df.copy()
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to group by a categorical field (string/object dtype)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean {numeric_field} by {group_field} (@id):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization
Visualize distributions or relationships between fields in the selected record set.

We'll display a histogram of a numerical field and, if available, a boxplot grouped by a category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df.empty or numeric_field is None:
    print("No visualization possible: DataFrame or numeric field missing.")
else:
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group field is available, show a boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and explore a Croissant-standardized dataset using the `mlcroissant` library. We accessed the FAIR² dataset's record sets and fields via their `@id` values, extracted tables into pandas DataFrames, and conducted basic exploratory data analysis—such as filtering, normalization, and simple visualizations. 

For deeper insights, consider consulting the dataset's documentation, reviewing variable definitions, and tailoring analytic steps to your research needs.

**Note:** The field and grouping choices in this notebook are illustrative and should be adapted according to the precise fields and record structures identified for your use case.